# Phase10~13 Verification Visualization
Figure text is English only. Korean explanations are provided via print().


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid', context='talk')

DATA_DIR = Path('../data/phase10_13')
ver_all = pd.read_csv(DATA_DIR / 'verification_all_dedup.csv')
ver_main = pd.read_csv(DATA_DIR / 'verification_main_h3_n20.csv')
ver_support = pd.read_csv(DATA_DIR / 'verification_support_p10_n10.csv')
seed_stats = pd.read_csv(DATA_DIR / 'verification_setting_seed_stats.csv')
router_scalar = pd.read_csv(DATA_DIR / 'router_stage_scalar.csv')
router_family = pd.read_csv(DATA_DIR / 'router_family_expert_long.csv')
router_pos = pd.read_csv(DATA_DIR / 'router_position_expert_long.csv')

print('verification all/main/support row:', len(ver_all), len(ver_main), len(ver_support))
print('main phase counts:', ver_main.groupby('source_phase').size().to_dict())
print('diag missing in main:', ver_main.loc[ver_main['diag_available'] != 1, 'run_phase'].tolist())


In [ ]:
print('setting 평균(best valid) heatmap입니다. phase별 어떤 setting이 상위권인지 빠르게 비교합니다.')

seed_main = seed_stats[(seed_stats['hparam_id'] == 'H3') & (seed_stats['source_phase'].isin(['P11', 'P12', 'P13']))].copy()
mat = seed_main.pivot_table(index='setting_key', columns='source_phase', values='best_valid_mean', aggfunc='mean')

plt.figure(figsize=(9.5, 11.5))
sns.heatmap(mat, annot=True, fmt='.4f', cmap='YlGnBu')
plt.title('Verification H3 Mean Best Valid')
plt.xlabel('Phase')
plt.ylabel('Setting Key')
plt.tight_layout()
plt.show()


In [ ]:
print('seed 분포(box+strip)로 setting 내부 분산을 봅니다. 평균뿐 아니라 분산을 같이 보는 것이 verification의 핵심입니다.')

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))

sns.boxplot(data=ver_main, x='source_phase', y='best_valid_mrr20', ax=axes[0], color='#4C78A8')
sns.stripplot(data=ver_main, x='source_phase', y='best_valid_mrr20', ax=axes[0], color='black', alpha=0.45, size=4)
axes[0].set_title('Best Valid Distribution by Phase')
axes[0].set_xlabel('Phase')
axes[0].set_ylabel('Best Valid MRR@20')

sns.boxplot(data=ver_main, x='source_phase', y='test_mrr20', ax=axes[1], color='#F58518')
sns.stripplot(data=ver_main, x='source_phase', y='test_mrr20', ax=axes[1], color='black', alpha=0.45, size=4)
axes[1].set_title('Test Distribution by Phase')
axes[1].set_xlabel('Phase')
axes[1].set_ylabel('Test MRR@20')

plt.tight_layout()
plt.show()


In [ ]:
print('mean-std scatter로 성능-안정성 trade-off를 봅니다. 오른쪽 위가 아니라 오른쪽 아래(높은 mean, 낮은 std)가 이상적입니다.')

tmp = seed_main.copy()

fig, ax = plt.subplots(figsize=(9.5, 6.0))
sns.scatterplot(data=tmp, x='best_valid_mean', y='best_valid_std', hue='source_phase', style='source_phase', s=120, ax=ax)
ax.set_title('Mean-Std Trade-off (Best Valid)')
ax.set_xlabel('Mean Best Valid MRR@20')
ax.set_ylabel('Std of Best Valid MRR@20')

for phase in ['P11', 'P12', 'P13']:
    sub = tmp[tmp['source_phase'] == phase]
    if sub.empty:
        continue
    top = sub.sort_values('best_valid_mean', ascending=False).head(1)
    for _, r in top.iterrows():
        ax.text(r['best_valid_mean'], r['best_valid_std'], r['setting_key'], fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
print('special slice와 test를 함께 봅니다. test 상위 setting이 cold/long에서도 일관적으로 좋은지 확인합니다.')

top = seed_main.sort_values('test_mean', ascending=False).head(10).copy()
plot_df = top[['setting_key', 'test_mean', 'cold_mean', 'long_session_mean']].melt(id_vars='setting_key', var_name='metric', value_name='value')

plt.figure(figsize=(13.2, 5.6))
sns.barplot(data=plot_df, x='setting_key', y='value', hue='metric')
plt.title('Top Settings: Test / Cold / Long Means')
plt.xlabel('Setting Key')
plt.ylabel('MRR@20')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
print('diag vs 성능 산점도입니다. concentration(top1), dispersion(cv), n_eff가 성능과 같이 어떻게 움직이는지 확인합니다.')

diag = ver_main[ver_main['diag_available'] == 1].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))

sns.scatterplot(data=diag, x='diag_top1_max_frac', y='best_valid_mrr20', hue='source_phase', style='setting_group', s=90, ax=axes[0])
axes[0].set_title('Top1 Concentration vs Best Valid')
axes[0].set_xlabel('Diag Top1 Max Fraction')
axes[0].set_ylabel('Best Valid MRR@20')

sns.scatterplot(data=diag, x='diag_cv_usage', y='test_mrr20', hue='source_phase', style='setting_group', s=90, ax=axes[1])
axes[1].set_title('Usage CV vs Test')
axes[1].set_xlabel('Diag CV Usage')
axes[1].set_ylabel('Test MRR@20')

axes[0].legend(loc='best', fontsize=8)
axes[1].legend(loc='best', fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
print('상관 heatmap으로 metric 관계를 정리합니다. 계획 문서에서 의도한 diag 해석 축을 한 번에 점검합니다.')

cols = [
    'best_valid_mrr20', 'test_mrr20', 'cold_item_mrr20', 'long_session_mrr20',
    'diag_top1_max_frac', 'diag_cv_usage', 'diag_n_eff',
    'diag_route_consistency_feature_group_knn_mean_score'
]
use = diag[cols].copy()
corr = use.corr(numeric_only=True)

plt.figure(figsize=(8.4, 6.8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0)
plt.title('Metric Correlation (Verification Main)')
plt.tight_layout()
plt.show()


In [ ]:
print('router family usage를 setting group 단위로 평균 냅니다. 어떤 axis가 어떤 family에 더 의존하는지 보는 셀입니다.')

main_keys = set(seed_main['setting_key'].tolist())
rf = router_family[(router_family['hparam_id'] == 'H3') & (router_family['setting_key'].isin(main_keys)) & (router_family['stage_name'] == 'macro@1')].copy()

if rf.empty:
    print('router family rows are empty for selected filters.')
else:
    fam = (rf.groupby(['source_phase', 'setting_group', 'family'], as_index=False)['family_expert_share_norm']
             .mean())

    g = sns.catplot(
        data=fam,
        x='family', y='family_expert_share_norm',
        hue='setting_group', col='source_phase',
        kind='bar', col_wrap=3, height=4.2, aspect=1.05,
        sharey=False
    )
    g.set_titles('{col_name} Family Usage by Group')
    g.set_axis_labels('Family', 'Mean Family-Expert Share')
    for ax in g.axes.flat:
        ax.tick_params(axis='x', rotation=30)
    plt.tight_layout()
    plt.show()


In [ ]:
print('router position usage를 통해 position별 집중도를 봅니다. position마다 top expert share가 올라가면 경직된 routing 가능성이 있습니다.')

rp = router_pos[(router_pos['hparam_id'] == 'H3') & (router_pos['setting_key'].isin(main_keys)) & (router_pos['stage_name'] == 'macro@1')].copy()

if rp.empty:
    print('router position rows are empty for selected filters.')
else:
    pos_top = (rp.groupby(['source_phase', 'setting_group', 'run_phase', 'position_index'], as_index=False)['position_expert_share_norm']
                 .max())
    pos_mean = (pos_top.groupby(['source_phase', 'setting_group', 'position_index'], as_index=False)['position_expert_share_norm']
                      .mean())

    g = sns.FacetGrid(pos_mean, col='source_phase', hue='setting_group', col_wrap=3, height=4.0, sharey=False)
    g.map_dataframe(sns.lineplot, x='position_index', y='position_expert_share_norm')
    g.add_legend()
    g.set_titles('{col_name} Position Concentration')
    g.set_axis_labels('Position Index', 'Mean Top Expert Share')
    plt.tight_layout()
    plt.show()


In [ ]:
print('router scalar를 묶어서 setting group별 요약을 봅니다. consistency score와 concentration(top1/cv)을 같이 두면 trade-off 해석이 쉬워집니다.')

rs = router_scalar[(router_scalar['hparam_id'] == 'H3') & (router_scalar['setting_key'].isin(main_keys)) & (router_scalar['stage_name'] == 'macro@1')].copy()

if rs.empty:
    print('router scalar rows are empty for selected filters.')
else:
    g = (rs.groupby(['source_phase', 'setting_group'], as_index=False)
           .agg(route_score=('route_consistency_feature_group_knn_mean_score', 'mean'),
                top1=('top1_max_frac', 'mean'),
                cv=('cv_usage', 'mean')))

    fig, ax = plt.subplots(figsize=(10.2, 6.0))
    sns.scatterplot(data=g, x='route_score', y='top1', hue='source_phase', style='setting_group', size='cv', sizes=(60, 260), ax=ax)
    ax.set_title('Router Consistency vs Concentration')
    ax.set_xlabel('Feature Group KNN Mean Score')
    ax.set_ylabel('Top1 Max Fraction')
    plt.tight_layout()
    plt.show()

    print('요약 테이블:')
    print(g.sort_values(['source_phase', 'route_score'], ascending=[True, False]).to_string(index=False))


In [ ]:
print('최종 추천 요약입니다. phase별로 valid/test/stability 관점에서 서로 다른 winner가 나올 수 있으므로 용도별 선택이 필요합니다.')

for phase in ['P11', 'P12', 'P13']:
    sub = seed_main[seed_main['source_phase'] == phase].copy()
    if sub.empty:
        continue
    best_valid = sub.sort_values('best_valid_mean', ascending=False).iloc[0]
    best_test = sub.sort_values('test_mean', ascending=False).iloc[0]
    best_stable = sub.sort_values('best_valid_std', ascending=True).iloc[0]

    print('')
    print('[{}]'.format(phase))
    print(' valid 최고:', best_valid['setting_key'], ' / ', round(float(best_valid['best_valid_mean']), 4), '+-', round(float(best_valid['best_valid_std']), 4))
    print(' test  최고:', best_test['setting_key'], ' / ', round(float(best_test['test_mean']), 4), '+-', round(float(best_test['test_std']), 4))
    print(' 안정성  :', best_stable['setting_key'], ' / std=', round(float(best_stable['best_valid_std']), 4))
